# Experiment run — packed: luyện dùng API của `qtomo_skeleton`

Cùng thí nghiệm với `experiment_run.ipynb` (học một unitary Haar-random $\mathcal{E}$, ra **Fig. 2** và
**Fig. 3**), nhưng lần này viết bằng **API**: lớp `KrausTomography` bạn vừa hoàn thiện, cùng `writer` và `plot`.
Mỗi ô **TODO** thay một nhóm khối của bản unpack bằng vài lời gọi API:

| Bản unpack (`experiment_run.ipynb`) | Bản packed (notebook này) | TODO |
|---|---|---|
| `[B1]`–`[B3]` RNG, trạng thái train, Kraus khởi tạo | `KrausTomography(...)` | P1 |
| `[B5]`–`[B7]` loss, vòng lặp tối ưu, lưu kết quả | `tomo.fit(...)` | P2 |
| `[B8]` đánh giá | `tomo.evaluate(...)` / `Evaluation.from_model(...)` | P3, P4 |
| `[B9]` đóng gói | `TomographyResult.from_model(...)` | P4 |
| `run_unpacked(...)` | `run_packed(...)` | P5 |

**Điều kiện:** đã làm xong TODO 1–4 trong `src/qtomo_skeleton/tomography.py`
(`python -m pytest tests/test_skeleton.py` không còn SKIPPED).
Mỗi TODO có một ô **kiểm tra** ngay sau — chạy qua được thì làm tiếp.

Gợi ý: đọc docstring bằng `help(KrausTomography.fit)`, `help(TomographyResult.from_model)`, …
Chạy từ thư mục `workshop/`.

In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # ẩn log khởi động của TensorFlow

# Dùng qtomo_skeleton trong workshop/src. Đã `pip install -e .` thì có thể xoá 3 dòng dưới.
SRC = (Path.cwd() / "src").resolve()
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

%matplotlib inline
import numpy as np
import tensorflow as tf

from qtomo_skeleton import KrausTomography, generator, plot, quantum_ops, writer
from qtomo_skeleton.optimizer import AdamOptimizer
from qtomo_skeleton.writer import Evaluation, ExperimentWriter, TomographyResult

try:  # KrausTomography của bạn đã xong chưa?
    KrausTomography(1, 1, seed=0).is_fitted
except NotImplementedError as exc:
    raise RuntimeError(f"Làm TODO trong src/qtomo_skeleton/tomography.py trước ({exc}).") from None
print("KrausTomography từ", sys.modules[KrausTomography.__module__].__file__)

## Cấu hình
Giống `experiment_run.ipynb`: cùng seed thì kết quả phải **trùng** bản unpack.

In [ ]:
# Phần 1 — một run
NUM_QUBITS = 1
SEED = 100

# Phần 2 — quét
QUBITS = [1, 2]       # số qubit cần quét
RUNS = 3              # số run (seed khác nhau) cho mỗi số qubit

# Chung
NUM_RHO = 3           # số probe state để train
NUM_KRAUS = 1         # số toán tử Kraus
NUM_TEST = 5          # số trạng thái test
EPOCHS = 300
LR = 0.05

OUTPUT_DIR = Path("output") / "experiment_run_packed"
FIGURE_DIR = OUTPUT_DIR / "figures"

---
# Phần 1 — Một run bằng API

## Bài toán (cho sẵn — khối `[B4]`, nằm ngoài class)
Process đích $\mathcal{E}$, cách ghép $\mathcal{E}^\dagger K(\rho)\mathcal{E}$ (`data_process_fn`) và process tham chiếu
dùng khi đánh giá (`target_process_fn`). Class không biết gì về bài toán cụ thể — nó chỉ nhận các hàm này.

In [ ]:
experiment_rng = np.random.default_rng(SEED)        # RNG cho target và trạng thái test
target = generator.haar(NUM_QUBITS, experiment_rng)


def data_process_fn(rho_set, kraus_ops):
    return [
        quantum_ops.apply_unitary_dagger(quantum_ops.apply_kraus_operators(rho, kraus_ops), target)
        for rho in rho_set
    ]


def target_process_fn(rho):
    return quantum_ops.apply_unitary(rho, target)

## TODO P1 — dựng model  (thay `[B1]`–`[B3]`)
Tạo `tomo` với `NUM_QUBITS`, `NUM_RHO`, `NUM_KRAUS` và `seed=SEED`.
Câu hỏi: tham số nào là positional, tham số nào bắt buộc phải truyền theo tên? Vì sao?

In [ ]:
# TODO P1
tomo = ...

In [ ]:
# Kiểm tra P1
assert len(tomo.target_rho_set) == NUM_RHO and len(tomo.init_kraus) == NUM_KRAUS
assert not tomo.is_fitted
print("P1 ✓", len(tomo.target_rho_set), "trạng thái train,", len(tomo.init_kraus), "Kraus op")

## TODO P2 — train  (thay `[B5]`–`[B7]`)
Gọi `tomo.fit(...)` với `data_process_fn` ở trên, `EPOCHS` epoch và optimizer `AdamOptimizer(alpha=LR)`.
`fit` nhận tham số **keyword-only** và trả `(kraus_operators, loss_history)`.

In [ ]:
# TODO P2
kraus_operators, loss_history = ...

In [ ]:
# Kiểm tra P2
assert tomo.is_fitted and len(loss_history) == EPOCHS and tomo.loss_history == loss_history
print(f"P2 ✓ loss: {loss_history[0]:.3e} -> {loss_history[-1]:.3e}, {tomo.fit_seconds:.1f} s")

## TODO P3 — đánh giá từng trạng thái  (thay `[B8]`)
Với mỗi trạng thái test, gọi `tomo.evaluate(rho, target_process_fn)` và lưu **phần thực** của fidelity
(phần tử thứ 3 của bộ kết quả) vào `fidelities`.

In [ ]:
test_states = generator.haar_probe_states(NUM_QUBITS, NUM_TEST, experiment_rng)
fidelities = []
for rho in test_states:
    # TODO P3
    ...

In [ ]:
# Kiểm tra P3
assert len(fidelities) == NUM_TEST
print("P3 ✓ fidelity:", np.round(fidelities, 6), " mean =", np.mean(fidelities))

## TODO P4 — đóng gói kết quả  (thay `[B9]`)
Vòng for ở P3 là việc lặp lại ở mọi thí nghiệm, nên `writer` đã gói sẵn:
- `Evaluation.from_model(tomo, states, target_process_fn)` → `evaluation`;
- `TomographyResult.from_model(tomo, evaluation, **metadata)` → `result`, với metadata
  `seed`, `epochs`, `lr`, `num_rho`, `num_kraus`.

So với `[B9]` của bản unpack: bạn không phải tự `np.stack` từng mảng nữa — vì sao `from_model` làm được việc đó?

In [ ]:
# TODO P4
evaluation = ...
result = ...

In [ ]:
# Kiểm tra P4
assert np.allclose(evaluation.fidelities, fidelities)
assert result.params.shape == (NUM_KRAUS, 2**NUM_QUBITS, 2**NUM_QUBITS)
assert result.metadata["seed"] == SEED
print(f"P4 ✓ final loss = {result.final_loss:.2e}, test infidelity = {result.test_infidelity:.2e}")

---
# Phần 2 — Một hàm cho một run, rồi quét

## TODO P5 — `run_packed`
Viết lại `run_unpacked` của `experiment_run.ipynb` bằng API: ~10 dòng thay vì ~40. Trả một `TomographyResult`
có metadata `seed`, `run`, `epochs`, `lr`, `num_rho`, `num_kraus`.

Giữ **đúng thứ tự dùng RNG** như bản unpack: target lấy từ `experiment_rng` trước, trạng thái test sau;
model tự tạo RNG riêng từ `seed`.

In [ ]:
def run_packed(num_qubits, seed, run=0):
    experiment_rng = np.random.default_rng(seed)
    target = generator.haar(num_qubits, experiment_rng)

    def data_process_fn(rho_set, kraus_ops):
        return [quantum_ops.apply_unitary_dagger(quantum_ops.apply_kraus_operators(r, kraus_ops), target)
                for r in rho_set]

    def target_process_fn(rho):
        return quantum_ops.apply_unitary(rho, target)

    # TODO P5: model -> fit -> trạng thái test (từ experiment_rng) -> Evaluation -> TomographyResult
    ...

In [ ]:
# Kiểm tra P5: cùng seed thì phải trùng Phần 1
check = run_packed(NUM_QUBITS, SEED)
assert isinstance(check, TomographyResult), "run_packed phải trả TomographyResult"
assert np.allclose(check.loss_history, result.loss_history), "run_packed lệch với Phần 1"
assert check.metadata["run"] == 0
print("P5 ✓ run_packed khớp Phần 1")

## Quét và lưu bằng `ExperimentWriter`
Tạo `exp = ExperimentWriter(OUTPUT_DIR, overwrite=True)`. Với mỗi `num_qubits` trong `QUBITS` và `run` trong
`range(RUNS)`: `seed = 100 * num_qubits + run`, chạy `run_packed`, lưu bằng `exp.save(result, name=...)` với tên
`f"{num_qubits}q_run{run}"`. In một dòng tóm tắt mỗi run (`final_loss`, `test_infidelity`, `fit_seconds`).

In [ ]:
exp = ExperimentWriter(OUTPUT_DIR, overwrite=True)
for num_qubits in QUBITS:
    for run in range(RUNS):
        seed = 100 * num_qubits + run
        res = run_packed(num_qubits, seed, run=run)
        path = exp.save(res, name=f"{num_qubits}q_run{run}")
        print(f"{path.name}: train loss = {res.final_loss:.2e}, "
              f"test infidelity = {res.test_infidelity:.2e}, {res.fit_seconds:.1f} s")

---
# Phần 3 — Đọc lại từ đĩa và vẽ hình

## Đọc lại kết quả

In [ ]:
results = [r for r in writer.load_results(OUTPUT_DIR, pattern="*q_run*.npz")
           if r.num_qubits in QUBITS and r.metadata["run"] < RUNS]
for num_qubits, runs in writer.group_by(results, "num_qubits").items():
    best = min(r.final_loss for r in runs)
    mean_test = np.mean([r.test_infidelity for r in runs])
    print(f"{num_qubits} qubit(s): best train loss = {best:.2e}, mean test infidelity = {mean_test:.2e}")

## Fig. 2: fidelity theo iteration

In [ ]:
fig2 = plot.plot_fidelity_curves(results)
plot.save_figure(fig2, FIGURE_DIR / "fig2", formats=("png", "eps"))
fig2

## Fig. 3: ma trận mật độ $\rho_\mathcal{E}$ vs $\rho_k$

In [ ]:
best = min((r for r in results if r.num_qubits == 1), key=lambda r: r.final_loss)
fig3 = plot.plot_density_matrices({
    r"$\rho_\mathcal{E}$": best.evaluation.targets[0],
    r"$\rho_k$": best.evaluation.outputs[0],
})
plot.save_figure(fig3, FIGURE_DIR / "fig3", formats=("png", "eps"))
fig3

## Kiểm tra cuối

In [ ]:
one_qubit = [r for r in results if r.num_qubits == 1]
assert min(r.final_loss for r in one_qubit) < 0.05, "Không run 1-qubit nào hội tụ"
assert all(np.all(np.isfinite(r.loss_history)) for r in results), "Loss có NaN/inf"
assert (FIGURE_DIR / "fig2.png").exists() and (FIGURE_DIR / "fig3.png").exists()
print("OK — packed flow chạy thành công")

---
## So sánh hai cách viết
Đặt `experiment_run.ipynb` cạnh notebook này. Cùng một thí nghiệm, cùng kết quả đến từng bit — nhưng:
- bản packed **ngắn hơn** và đọc như mô tả thí nghiệm (dựng model → fit → evaluate → lưu → vẽ);
- chi tiết thuật toán (stack/unstack, chiếu CPTP, reset optimizer, lấy phần thực…) nằm **một chỗ** trong
  `KrausTomography`, có test, không bị copy lại ở mỗi notebook;
- trạng thái của một run sống trong `tomo`, nên có thể `fit` lại, `evaluate` thêm trạng thái mới, hay lưu
  bằng `TomographyResult.from_model(tomo, ...)` mà không phải nhớ tên từng biến.